# Підготовка даних
Семпл 100 тис. замовлень → поділ **користувачів** на train / val / test → parquet.

Запуск цілком з терміналу:
```
.venv/Scripts/jupyter nbconvert --to notebook --execute --inplace scripts/prepare_data.ipynb
```

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed"

SEED = 47
N_ORDERS = 100000
TEST_FRAC = 0.2
VAL_FRAC = 0.1

## Сирі дані

In [21]:
items_order = pd.read_csv(RAW / "order_products__prior.csv", dtype="int32")
order_to_user = pd.read_csv(RAW / "orders.csv", usecols=["order_id", "user_id"], dtype="int32")

In [22]:
display(items_order.head())
display(order_to_user.head())

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


,order_id,user_id
0,2539329,1
1,2398795,1
2,473747,1
3,2254736,1
4,431534,1


## Семпл
Обираємо номери замовлень і беремо **всі** їхні товари (кошики не обрізаємо).

In [23]:
rng = np.random.default_rng(SEED)
sampled = rng.choice(items_order["order_id"].unique(), size=N_ORDERS, replace=False)
baskets = items_order[items_order["order_id"].isin(sampled)].merge(order_to_user, on="order_id")

In [24]:
display(baskets.shape)
display(baskets.head())

(1005310, 5)

,order_id,product_id,add_to_cart_order,reordered,user_id
0,14,20392,1,1,18194
1,14,27845,2,1,18194
2,14,162,3,1,18194
3,14,2452,4,1,18194
4,14,8575,5,1,18194


## Поділ за користувачами

In [25]:
users = rng.permutation(np.sort(baskets["user_id"].unique()))
n_test = int(len(users) * TEST_FRAC)
n_val = int((len(users) - n_test) * VAL_FRAC)

split_of_user = pd.Series("train", index=users)
split_of_user[users[:n_test]] = "test"
split_of_user[users[n_test:n_test + n_val]] = "val"
baskets["split"] = baskets["user_id"].map(split_of_user)

baskets.groupby("split").agg(users=("user_id", "nunique"), orders=("order_id", "nunique"))

,users,orders
split,,
test,13474,19788
train,48509,72141
val,5389,8071


## Довідник товарів

In [26]:
products = (
    pd.read_csv(RAW / "products.csv")
    .merge(pd.read_csv(RAW / "aisles.csv"), on="aisle_id")
    .merge(pd.read_csv(RAW / "departments.csv"), on="department_id")
    [["product_id", "product_name", "aisle", "department"]]
)

In [27]:
products.head()

,product_id,product_name,aisle,department
0,1,Chocolate Sandwich Cookies,cookies cakes,snacks
1,2,All-Seasons Salt,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,frozen meals,frozen
4,5,Green Chile Anytime Sauce,marinades meat preparation,pantry


## Зберігаємо

In [28]:
OUT.mkdir(parents=True, exist_ok=True)
baskets.sort_values(["order_id", "add_to_cart_order"]).to_parquet(OUT / "baskets.parquet", index=False)
products.to_parquet(OUT / "products.parquet", index=False)